# CNO-sim undertrained check (accelerate)

Question: is shipped `sim_cno.pth` undertrained? Resume it **on `train_sim` itself** (same distribution) via `scripts/finetune_baseline.py` + `accelerate launch`. If val loss keeps dropping well below the shipped point, it was undertrained; if flat, it converged.

Optional Run B resumes the same ckpt on `train_real/` (finetune reference). Scoring uses the `baseline_kaggle.ipynb` scorer. **Runs on Kaggle GPU.**

In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR} ]; then echo "Pulling {REPO_DIR}..."; cd {REPO_DIR} && git pull; else echo "Cloning {REPO_URL}..."; git clone {REPO_URL} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}

In [ ]:
# Enforce Python 3.10 (starterkit)
!uv python pin 3.10 2>&1 | tail -1
!uv sync --python 3.10

In [ ]:
import sys, os
_REPO = os.environ.get('REPO_DIR', '/kaggle/working/realpde')
!{sys.executable} -m pip install -q -e . --no-deps --ignore-requires-python || echo install-failed-falling-back-to-src
for _p in (_REPO + '/src', _REPO):
    sys.path.insert(0, _p) if _p not in sys.path else None
import realpde; print('realpde OK ->', realpde.__file__)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '', '| count:', torch.cuda.device_count())

In [ ]:
# =============================================================================
# CONFIG — edit here only. Script reads everything from env.
# =============================================================================
import os
from pathlib import Path

ROOT_CANDIDATES = ['/kaggle/input/datasets/nthday/realpde', './data', 'data']  # Kaggle: {baseline,test,train_real,train_sim}/ directly underneath
DATA_ROOT = next((c for c in ROOT_CANDIDATES if Path(c).exists()), ROOT_CANDIDATES[0])
os.environ['DATA_ROOT'] = DATA_ROOT
root = Path(DATA_ROOT)

def resolve_h5_dir(*names):
    for name in names:
        base = root / name
        if not base.exists():
            continue
        if sorted(base.glob('*.h5')):
            return base
        nested = sorted(base.rglob('*.h5'))
        if nested:
            from collections import Counter
            return Counter(p.parent for p in nested).most_common(1)[0][0]
    return None

SIM_DIR, REAL_TR = resolve_h5_dir('train_sim'), resolve_h5_dir('train_real')
assert (root / 'baseline').exists() and (root / 'test').exists(), f'DATA_ROOT={root} lacks baseline/test — attach the realpde dataset'
print('SIM :', SIM_DIR)
print('REAL:', REAL_TR)

# Shipped sim CNO under test
cands = sorted((root / 'baseline').rglob('*cno*.pt*')) if (root / 'baseline').exists() else []
if not cands and Path('data/baseline_checkpoints').exists():
    cands = sorted(Path('data/baseline_checkpoints').rglob('*cno*.pt*'))
sim_cno = next((p for p in cands if 'sim_real' not in p.name.lower()), cands[0] if cands else None)
print('sim_cno:', sim_cno)
os.environ['SIM_CNO'] = str(sim_cno)

# Shared hyperparams (finetune LR < pretrain 1e-3)
os.environ.update({'MODEL_TYPE': 'cno', 'IN_STEP': '20', 'OUT_STEP': '20', 'INTERVAL': '20',
    'SUB_S': '2', 'LR': '1e-4', 'EPOCHS': '20', 'BATCH_SIZE': '4',
    'VAL_FRAC': '0.1', 'SEED': '42', 'NUM_WORKERS': '2', 'WANDB_PROJECT': 'realpde-finetune'})
# Wandb: pull API key from Kaggle secret WANDB_KEY (Attachments -> Secrets);
# without it, wandb runs disabled so training still works.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_KEY')
    print('WANDB_KEY loaded from Kaggle secrets')
except Exception as e:
    os.environ.setdefault('WANDB_MODE', 'disabled')
    print(f'WANDB_KEY secret not set ({e}) - wandb disabled')
print('config OK')

In [ ]:
# --- Run A (the undertrained test): resume sim_cno ON train_sim ---
# Same distribution as its pretraining. Falling val = was undertrained.
import os, torch
os.environ['DATA_PATH'] = str(SIM_DIR)
os.environ['RESUME_CKPT'] = os.environ['SIM_CNO']
os.environ['SAVE_DIR'] = '/kaggle/working/cno_sim_resume'
os.environ['WANDB_RUN_NAME'] = 'cno-sim-resume'
mixed = 'fp16' if torch.cuda.is_available() else 'no'
nproc = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"DATA_PATH={os.environ['DATA_PATH']} SAVE_DIR={os.environ['SAVE_DIR']} procs={nproc}")
!uv run --python 3.10 accelerate launch --mixed_precision={mixed} --num_processes={nproc} --num_machines=1 --dynamo_backend=no scripts/finetune_baseline.py

In [ ]:
# --- Run B (optional): same ckpt finetuned on train_real ---
import os, torch
os.environ['DATA_PATH'] = str(REAL_TR)
os.environ['RESUME_CKPT'] = os.environ['SIM_CNO']
os.environ['SAVE_DIR'] = '/kaggle/working/cno_real_ft'
os.environ['WANDB_RUN_NAME'] = 'cno-real-ft'
mixed = 'fp16' if torch.cuda.is_available() else 'no'
nproc = torch.cuda.device_count() if torch.cuda.is_available() else 1
!uv run --python 3.10 accelerate launch --mixed_precision={mixed} --num_processes={nproc} --num_machines=1 --dynamo_backend=no scripts/finetune_baseline.py

In [ ]:
# --- Verdict: shipped sim_cno vs continued best/final on the SAME sim val split ---
# Same scorer formula as baseline_kaggle.ipynb (raw-space MSE + rel-L2).
# `continued << shipped` on sim = undertrained. `continued ~= shipped` = converged.
import gc, os
from pathlib import Path
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm
from realpde.datasets import PDEDataset
from load_baseline import load_baseline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ds = PDEDataset(SIM_DIR, in_step=20, out_step=20, interval=20, sub_s=2)
n_val = max(1, int(0.1 * len(ds)))
_, val_ds = random_split(ds, [len(ds) - n_val, n_val], generator=torch.Generator().manual_seed(42))
loader = DataLoader(val_ds, batch_size=4, shuffle=False)

def score(path):
    m, _ = load_baseline(str(path), device=DEVICE)
    m.eval()
    ms, rs, n = 0.0, 0.0, 0
    with torch.no_grad():
        for inp, tgt in tqdm(loader, desc=Path(path).name, leave=False):
            inp, tgt = inp.to(DEVICE), tgt.to(DEVICE)
            pred = m(inp)
            ms += F.mse_loss(pred, tgt, reduction='sum').item()
            b = pred.shape[0]
            p, t = pred.reshape(b, -1), tgt.reshape(b, -1)
            rs += ((p - t).norm(dim=1) / t.norm(dim=1).clamp_min(1e-8)).sum().item()
            n += b
    del m; gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return ms / max(n * 20 * 32 * 64 * 3, 1), rs / max(n, 1), n

paths = {'shipped-sim-cno': Path(os.environ['SIM_CNO']),
           'continued-best': Path('/kaggle/working/cno_sim_resume/best.pth'),
           'continued-final': Path('/kaggle/working/cno_sim_resume/final.pth')}
for name, p in paths.items():
    if p.exists():
        mse, rel, n = score(p)
        print(f'{name:16s} MSE {mse:.6f}  rel-L2 {rel:.6f}  (n={n})')
    else:
        print(f'{name:16s} MISSING ({p})')